# 💻 NB-12: CodeT5 Fine-tuning – Extract & Generate Code
**Goal:** Extract code blocks from Claude responses and fine-tune CodeT5 for code generation from natural language.

**Modality:** Code Generation | **Model:** Salesforce/codet5-small

In [ ]:
!pip install transformers datasets -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import re
from transformers import RobertaTokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import torch

# Extract code blocks from responses
code_pairs = []
for d in data:
    blocks = re.findall(r"```(?:python|javascript|bash|java|cpp|c\+\+|sql)?
(.*?)```", d["response"], re.DOTALL)
    for block in blocks:
        if len(block.strip()) > 30:
            code_pairs.append({"desc": d["user"], "code": block.strip()[:512]})

print(f"Extracted {len(code_pairs)} code snippets")

MODEL = "Salesforce/codet5-small"
tok = RobertaTokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL)

def preprocess(batch):
    inp = tok(["Generate code: " + d for d in batch["desc"]], max_length=128, truncation=True, padding="max_length")
    tgt = tok(batch["code"], max_length=256, truncation=True, padding="max_length")
    inp["labels"] = [[(t if t != tok.pad_token_id else -100) for t in l] for l in tgt["input_ids"]]
    return inp

ds = Dataset.from_dict({"desc": [p["desc"] for p in code_pairs],
                         "code": [p["code"] for p in code_pairs]})
ds = ds.map(preprocess, batched=True, remove_columns=["desc","code"]).train_test_split(0.1)

args = Seq2SeqTrainingArguments("./codet5-claude", num_train_epochs=5,
    per_device_train_batch_size=8, predict_with_generate=True,
    fp16=torch.cuda.is_available(), evaluation_strategy="epoch", report_to="none")
Seq2SeqTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
    tokenizer=tok, data_collator=DataCollatorForSeq2Seq(tok, model)).train()
print("✅ CodeT5 fine-tuned on Claude code examples!")


In [ ]:
# --- Inference ---
prompt = "Write a Python function that calculates fibonacci numbers"
inp = tok("Generate code: " + prompt, return_tensors="pt").input_ids
out = model.generate(inp, max_new_tokens=150, num_beams=4)
print(tok.decode(out[0], skip_special_tokens=True))
